# Lecture 3 live demo: reading real PyTorch traces
HPML | Fall 2026 | Dr. Kaoutar El Maghraoui

Each section produces a trace file you open in **ui.perfetto.dev** (or chrome://tracing) and a table you read in the notebook.

1. Setup and a helper that profiles any function
2. Eager vs compiled ResNet-18: the before/after in the trace
3. Launch bound: a Python loop against a vectorized version
4. Hidden sync: `.item()` every step against once at the end
5. Input pipeline: `num_workers=0` against workers and pinned memory
6. Optimizer: for-loop, foreach and fused AdamW
7. Memory snapshot for pytorch.org/memory_viz
8. Optional: HTA summary of a trace folder

Run on a GPU runtime. Numbers depend on your GPU, CPU and PyTorch version.

### How to use this notebook on your own

**You do not need a GPU to learn from this.** Every cell already has its output saved, and every profiler run has a matching trace file in `traces/`. So you can read the whole thing without running anything.

**Reading a trace:** go to **ui.perfetto.dev** and drag one of the `traces/*.json.gz` files onto the page (Perfetto opens the compressed files directly; no need to unzip). The section right below ("How to read a trace in Perfetto") explains what you are looking at. Start with `traces/launch_bound_loop.json.gz` next to `traces/launch_bound_vectorized.json.gz`: the difference is the easiest one to see.

**Suggested order:** read section by section, top to bottom. For each one, first read the "Look for" notes, then open the two trace files it produced and find those signatures yourself. The **Explore next** prompts are optional experiments you can try if you re-run.

**Re-running (optional):** you need a GPU runtime (Colab's free T4 is enough). Your numbers will differ from the saved A100 numbers, but the *shapes* in the trace will be the same. Run the cells in order, since later sections reuse things defined earlier.

**The memory snapshot:** section 7 writes `mem.pickle`; drag it onto **pytorch.org/memory_viz** to see the allocation timeline.

In [1]:
# Standard imports. torch is the core library; torchvision gives us ready-made models.
import os, time, torch, torchvision
# The profiler lives in torch.profiler. We import the pieces we use by name.
from torch.profiler import profile, schedule, ProfilerActivity, record_function

# Every demo needs a GPU. This stops early with a clear message if none is attached.
assert torch.cuda.is_available(), "Use a GPU runtime"
dev = "cuda"                                   # we put tensors and models on the GPU

# Allow TF32 math on the tensor cores. It makes float32 matmuls and convolutions much
# faster on Ampere and newer GPUs, with a tiny precision loss that is fine for training.
torch.set_float32_matmul_precision("high")

print(torch.__version__, torch.cuda.get_device_name(0))   # note versions for reproducibility
os.makedirs("traces", exist_ok=True)           # folder to hold the trace files we export

2.14.0+cu130 NVIDIA A100-SXM4-40GB


## 1. One helper for every demo
Warm up, then profile a few steps with a schedule. It saves a trace and prints the top rows sorted by self CUDA time or self CPU time.

In [2]:
def run_profile(step_fn, name, steps=6, sort="self_cuda_time_total", rows=12, warmup_calls=3, **kw):
    # Run step_fn a few times BEFORE profiling. The first calls trigger one-time setup
    # (memory allocation, cuDNN picking algorithms, compilation) that we do not want to
    # measure. This is called "warming up".
    for _ in range(warmup_calls):
        step_fn()
    torch.cuda.synchronize()                   # wait for the GPU to finish the warmup

    # The schedule says which steps to actually record: skip 1 (wait), ignore 1 more while
    # things settle (warmup), then record the next `active` steps, and do this once (repeat=1).
    sched = schedule(wait=1, warmup=1, active=steps - 2, repeat=1)
    path = f"traces/{name}.json"

    # activities=CPU records the Python/operator side; CUDA records the GPU kernels.
    # record_shapes stores each op's tensor shapes (useful, but it makes the file bigger).
    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
                 schedule=sched, record_shapes=True, **kw) as prof:
        for _ in range(steps):
            step_fn()
            prof.step()                        # tell the profiler one step just finished
        torch.cuda.synchronize()

    prof.export_chrome_trace(path)             # write the timeline you open in Perfetto
    # key_averages() sums every event by name; .table() prints that summary, sorted.
    print(prof.key_averages().table(sort_by=sort, row_limit=rows))
    print("trace:", path, "-> open in https://ui.perfetto.dev")
    return prof

def bench(fn, n=50):
    """Time fn accurately with CUDA events (and without the profiler's overhead)."""
    for _ in range(5): fn()                    # warm up so we do not time one-off setup
    # The GPU runs asynchronously, so a normal CPU stopwatch would be wrong. CUDA events
    # are markers placed on the GPU's own timeline; the gap between them is real GPU time.
    s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()       # start marker
    for _ in range(n): fn()
    e.record(); torch.cuda.synchronize()       # stop marker, then wait for the GPU to reach it
    return s.elapsed_time(e) / n               # total milliseconds / number of calls

## How to read a trace in Perfetto

Every `run_profile` call writes a `.json` file (the copies saved in `traces/` are gzipped as `.json.gz`, which Perfetto opens directly). Open one at **ui.perfetto.dev** (drag the file onto the page) to get a timeline. Here is how to read it.

**The rows (tracks), top to bottom**
- **Python / CPU threads.** The thread that runs your Python calls `aten::` operators, `cudaLaunchKernel`, and your own `record_function` labels (`data`, `forward`, ...). Backward runs on a separate autograd thread.
- **GPU streams.** One row per CUDA stream. The bars here are the kernels actually running on the device (`sm80_...`, `elementwise_kernel`, `ampere_...`). This row is where real GPU time is spent.

**Navigate.** `W` / `S` zoom in and out, `A` / `D` pan left and right. Click a bar to read its name and duration. Select a range (click and drag) and press `M` to pin a measured interval; its width is your wall-clock answer.

**Connect a launch to its kernel.** Click a `cudaLaunchKernel` on the CPU row. Perfetto draws a **flow arrow** (labeled `ac2g`, async CPU to GPU) to the kernel it started on the GPU row. This is how you prove which line of Python caused which kernel.

**What each shape means**
- **A gap on the GPU row** means the GPU is idle: something upstream is not feeding it (Python overhead, a sync, or slow data).
- **A long CPU op over an empty GPU row** means launch bound or waiting on data, not GPU bound.
- **A `cudaStreamSynchronize` or `cudaDeviceSynchronize` bar** means the CPU is blocked waiting for the GPU: harmless if the GPU is busy, wasteful if it forces the GPU idle.
- **A dense comb of tiny GPU bars with gaps between them** means many small kernels, dominated by launch overhead.

**Table to trace.** The printed table aggregates the same events. `Self CUDA` is time in kernels (the GPU rows). `Self CPU` is time in the operator on the CPU thread. When `CPU total` is far larger than `CUDA total`, you are CPU or launch bound, and the trace will show a busy CPU row above an idle GPU row.

## 2. Eager vs compiled ResNet-18

Run the same model two ways: **eager** (every operation dispatched one at a time) and **`torch.compile`** (regions traced once and fused into fewer, larger kernels).

**The honest before/after.** On a modern GPU the wall-clock win is modest for a convolutional network, roughly 1.2-1.3x, because convolutions already run on highly tuned cuDNN kernels that compile cannot improve much. The bigger, more reliable difference is in the *shape* of the trace.

**Compare the two traces.** The eager trace is a long row of individual `aten::*` ops on the CPU, each launching its own small kernel on the GPU. The compiled trace collapses each forward into a single `CompiledFunction` (and `CompiledFunctionBackward` on the autograd thread), with fewer, larger fused kernels and fewer `cudaLaunchKernel` calls. Click a kernel and follow the `ac2g` flow arrow back to the CPU call that launched it.

**Watch for:** the first compiled call takes seconds because it includes compilation; every call after that is fast.

In [3]:
# Build ResNet-18 and put it on the GPU. This "eager" model runs one operation at a time.
model = torchvision.models.resnet18().to(dev)
# torch.compile returns a wrapped model with the SAME weights. The first time it runs it
# traces the network and generates fused kernels, so that first call is slow; later calls
# are fast. This is the "compiled" version we compare against eager.
model_c = torch.compile(model)

# One fake batch of images shaped (batch, channels, height, width), already on the GPU.
x = torch.randn(64, 3, 224, 224, device=dev)

def step(m):
    # A forward pass, then a backward pass. out.sum() collapses the output to a single
    # number so we can call .backward() on it. We only care about timing here, so we do
    # not bother zeroing gradients between calls.
    m(x).sum().backward()

# Time the very first compiled call on its own, so you can see the one-time compile cost.
t0 = time.perf_counter()
step(model_c)
torch.cuda.synchronize()                        # wait for the GPU so the timer is honest
print(f"first compiled call (includes compile): {time.perf_counter() - t0:.1f} s")

# The honest before/after, measured without the profiler.
print(f"eager:    {bench(lambda: step(model)):.2f} ms/step")
print(f"compiled: {bench(lambda: step(model_c)):.2f} ms/step")

# Profile both so we can open the two traces and compare their shape.
run_profile(lambda: step(model),   "resnet18_eager",    steps=5)
run_profile(lambda: step(model_c), "resnet18_compiled", steps=5)

first compiled call (includes compile): 29.4 s


eager:    25.52 ms/step


compiled: 19.98 ms/step


USDT:2026-09-28 04:09:08 4:4 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-09-28 04:09:08 4:4 SyncActivityProfilerHandler.cpp:46] profiler_stop


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                             aten::convolution_backward        10.85%      13.880ms        15.91%      20.345ms     339.084us      29.751ms        34.57%      29.751ms     495.853us            60  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us      24.557ms        28.53%      24.557ms       8.186ms             3  
void cudn

USDT:2026-09-28 04:09:09 4:4 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-09-28 04:09:09 4:4 SyncActivityProfilerHandler.cpp:46] profiler_stop


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
## Call CompiledFxGraph fsih6lv5eq3o7624peqvkormz37o...         0.00%       0.000us         0.00%       0.000us       0.000us      41.909ms        59.61%      41.909ms      13.970ms             3  
                             aten::convolution_backward         7.78%       8.595ms        12.16%      13.440ms     224.000us      27.108ms        38.56%      27.108ms     451.804us            60  
## Call C

**What the numbers show:** compiled is modestly faster per step than eager here (roughly 1.2-1.3x), which is expected for a convolutional network whose heavy kernels are already well tuned. The first compiled call takes seconds because it includes compilation; only steps after that are fast. The clearer story is in the traces: eager forward is a long list of `aten::*` ops each launching a small kernel, while compiled forward runs as one `CompiledFunction` on the main thread and `CompiledFunctionBackward` on the autograd thread, with fewer and larger fused kernels. Open both traces and compare how busy the CPU row is and how many `cudaLaunchKernel` calls each makes.

## 3. Launch bound
Normalize each row of a 1024 x 1024 tensor. Left: a Python loop, one row at a time. Right: one vectorized expression.

**Look for:** a thin comb of short kernels with gaps between them (the loop launches thousands of tiny ops, and the GPU waits between each) against a couple of long kernels back to back (vectorized). In the table, the loop has `CPU total` far above `CUDA total` and thousands of `cudaLaunchKernel` calls; the vectorized version has a handful.

In [4]:
# A 1024 x 1024 matrix on the GPU. We normalize each row (divide it by its length).
x = torch.randn(1024, 1024, device=dev)

def loop_version():
    # The slow way: loop over rows in Python, normalizing one row at a time. Each row
    # launches its own tiny GPU kernels, so we pay launch overhead 1024 times.
    out = torch.empty_like(x)                  # allocate the output up front
    for r in range(x.shape[0]):                # x.shape[0] is the number of rows
        out[r] = x[r] / x[r].norm()            # x[r].norm() is the length of row r
    return out

def vec_version():
    # The fast way: one expression over the whole tensor. dim=1 takes the norm along each
    # row; keepdim=True keeps a (rows, 1) shape so it broadcasts cleanly in the division.
    return x / x.norm(dim=1, keepdim=True)

# Check the two versions agree before trusting the speed comparison.
torch.testing.assert_close(loop_version(), vec_version())
print(f"loop: {bench(loop_version, 5):.2f} ms   vectorized: {bench(vec_version):.3f} ms")

loop: 55.87 ms   vectorized: 0.029 ms


In [5]:
# The loop launches thousands of tiny ops, so profile just one step to keep the trace small.
run_profile(loop_version, "launch_bound_loop", steps=3, sort="self_cpu_time_total")
run_profile(vec_version, "launch_bound_vectorized", steps=6)

USDT:2026-09-28 04:09:11 4:4 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-09-28 04:09:11 4:4 SyncActivityProfilerHandler.cpp:46] profiler_stop


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*        31.14%      39.777ms        99.98%     127.709ms     127.709ms       0.000us         0.00%      10.541ms      10.541ms             1  
                               aten::linalg_vector_norm        16.52%      21.099ms        25.57%      32.657ms      31.891us       4.616ms        43.79%       4.616ms       4.507us          1024  
         

USDT:2026-09-28 04:09:13 4:4 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-09-28 04:09:13 4:4 SyncActivityProfilerHandler.cpp:46] profiler_stop


**Explore next:** change 1024 to 128 rows and re-run. The loop now launches about eight times fewer kernels, so its time drops roughly in proportion, while the vectorized version stays about the same. The absolute gap shrinks, but the loop is still launch bound: its time tracks the number of Python iterations, not the amount of math.

## 4. Hidden synchronization
A running metric summed over many small GPU ops, the kind of loop that computes an average loss or accuracy over batches. Version A reads `.item()` every iteration. Version B keeps the running total on the GPU and reads it once at the end.

Each `.item()` copies one number from the GPU to the CPU, and to do that the CPU must **wait** for the GPU to reach that point. Do it every iteration and the CPU and GPU can no longer run ahead of each other: they take turns instead of overlapping.

**Look for in the trace (version A):** on the CPU row, a repeating group of `aten::item`, `aten::_local_scalar_dense`, `cudaMemcpyAsync`, and a `cudaStreamSynchronize` bar, with a **gap on the GPU row** right after each one where the device sits idle. In version B those four disappear and the GPU row is packed. In the table, A lists thousands of `cudaStreamSynchronize` calls; B lists none.

**The honest caveat:** a single `loss.item()` per step on a large model is usually cheap, because the GPU is busy enough that the CPU waiting for it costs nothing (see the ResNet trace). The cost shows up when you read a GPU scalar inside a hot loop, as here. The trace teaches you to *find* the sync; benchmarking tells you whether that particular sync is worth removing.

In [6]:
# A 256 x 256 matrix on the GPU, and K reductions we sum into a running total.
x = torch.randn(256, 256, device=dev)
K = 1000

def metric_sync():                  # A: the slow pattern
    acc = 0.0                       # a plain Python float, which lives on the CPU
    for _ in range(K):
        # x.square().sum() runs on the GPU. .item() copies that one number back to the CPU,
        # which forces the CPU to WAIT for the GPU on every single iteration.
        acc += x.square().sum().item()
    return acc

def metric_nosync():                # B: the fast pattern
    acc = torch.zeros((), device=dev)   # keep the running total ON the GPU (a 0-dim tensor)
    for _ in range(K):
        acc += x.square().sum()          # stays on the GPU, so nothing has to wait
    return acc.item()                    # one copy back to the CPU, at the very end

# Confirm both compute the same number (tiny floating-point differences are expected).
a, b = metric_sync(), metric_nosync()
print(f"same answer? relative diff {abs(a - b) / b:.1e}")
print(f"A sync every iter: {bench(metric_sync, 20):.2f} ms   B one sync: {bench(metric_nosync, 20):.2f} ms")
# Each call runs K=1000 iterations, so profile a single step to keep the trace small.
run_profile(metric_sync, "sync_every_iter", steps=3, sort="self_cpu_time_total")
run_profile(metric_nosync, "one_sync", steps=3, sort="self_cpu_time_total")

same answer? relative diff 5.4e-06


A sync every iter: 52.00 ms   B one sync: 31.49 ms


USDT:2026-09-28 04:09:15 4:4 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-09-28 04:09:15 4:4 SyncActivityProfilerHandler.cpp:46] profiler_stop


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                              aten::sum        18.70%      18.348ms        28.57%      28.037ms      28.037us      10.631ms        71.88%      10.631ms      10.631us          1000  
                                       cudaLaunchKernel        16.51%      16.204ms        16.51%      16.204ms       8.102us       0.000us         0.00%       0.000us       0.000us          2000  
         

USDT:2026-09-28 04:09:18 4:4 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-09-28 04:09:18 4:4 SyncActivityProfilerHandler.cpp:46] profiler_stop


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                       cudaLaunchKernel        29.27%      17.237ms        29.27%      17.237ms       5.744us       0.000us         0.00%       0.000us       0.000us          3001  
                                              aten::sum        22.29%      13.128ms        33.80%      19.902ms      19.902us      13.550ms        72.87%      13.550ms      13.550us          1000  
         

In [7]:
# Find syncs automatically (experimental; does not catch every case).
# set_sync_debug_mode("warn") prints a UserWarning at every synchronizing call.
torch.cuda.set_sync_debug_mode("warn")
_ = x.square().sum().item()          # the .item() here triggers the warning
torch.cuda.set_sync_debug_mode("default")

/usr/local/lib/python3.11/site-packages/torch/cuda/__init__.py:1395: UserWarning: Synchronization debug mode is a prototype feature and does not yet detect all synchronizing operations (Triggered internally at /__w/pytorch/pytorch/torch/csrc/cuda/Module.cpp:1080.)
  torch._C._cuda_set_sync_debug_mode(debug_mode)
/tmp/ipykernel_4/3147803636.py:4: UserWarning: called a synchronizing CUDA operation (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:162.)
  _ = x.square().sum().item()          # the .item() here triggers the warning


## 5. Input pipeline
A synthetic dataset with a deliberately slow CPU transform. Compare `num_workers=0` with workers and pinned memory.

**Look for:** long `enumerate(DataLoader)#...__next__` spans on the CPU with an empty GPU row underneath (A), and the copy label `Pageable -> Device` against `Pinned -> Device`.

In [8]:
from torch.utils.data import Dataset, DataLoader

# A fake image dataset. __getitem__ returns one sample; the slow CPU transform makes the
# data pipeline, not the GPU, the bottleneck, which is what we want to see in the trace.
class SlowImages(Dataset):
    def __len__(self): return 10_000                 # pretend we have 10k images
    def __getitem__(self, idx):
        img = torch.rand(3, 224, 224)                # one random (channels, H, W) image on the CPU
        for _ in range(3):                           # stand-in for decode and augmentation
            img = torch.nn.functional.avg_pool2d(img[None], 3, 1, 1)[0]
        return img, idx % 1000                       # (image, a fake class label 0..999)

cnn = torchvision.models.resnet18().to(dev)          # a model to consume the batches
cnn_opt = torch.optim.SGD(cnn.parameters(), lr=0.01)

def make_step(loader):
    # Returns a function that runs one training step, pulling the next batch from `loader`.
    it = iter(loader)                                # an iterator we advance with next()
    def step():
        # record_function labels a region so it appears as a named bar in the trace.
        with record_function("data"):
            x, y = next(it)                          # fetch the next batch (this is the slow part)
            # Copy to the GPU. non_blocking lets the copy overlap with other work when the
            # source memory is pinned (see loader_b below).
            x = x.to(dev, non_blocking=True); y = y.to(dev, non_blocking=True)
        with record_function("forward"):
            loss = torch.nn.functional.cross_entropy(cnn(x), y)   # predictions -> scalar loss
        with record_function("backward"):
            loss.backward()                          # compute gradients
        with record_function("optimizer"):
            cnn_opt.step()                           # update the weights
            cnn_opt.zero_grad(set_to_none=True)      # clear gradients for the next step
    return step

ds = SlowImages()
# Version A: no worker processes, so loading and GPU compute cannot overlap.
loader_a = DataLoader(ds, batch_size=64, num_workers=0)
# Version B: 8 background workers load ahead; pin_memory speeds the CPU->GPU copy;
# persistent_workers keeps workers alive between epochs; prefetch_factor queues batches.
loader_b = DataLoader(ds, batch_size=64, num_workers=8, pin_memory=True,
                      persistent_workers=True, prefetch_factor=2)
run_profile(make_step(loader_a), "input_workers0", steps=6, sort="self_cpu_time_total", warmup_calls=2)
run_profile(make_step(loader_b), "input_workers8_pinned", steps=6, sort="self_cpu_time_total", warmup_calls=4)

USDT:2026-09-28 04:09:21 4:4 SyncActivityProfilerHandler.cpp:39] profiler_start


USDT:2026-09-28 04:09:21 4:4 SyncActivityProfilerHandler.cpp:46] profiler_stop


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                       aten::avg_pool2d        40.49%     358.160ms        40.49%     358.160ms     466.355us       0.000us         0.00%       0.000us       0.000us           768  
                                         aten::uniform_        32.29%     285.580ms        32.29%     285.580ms       1.116ms       0.000us         0.00%       0.000us       0.000us           256  
         

USDT:2026-09-28 04:09:26 4:4 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-09-28 04:09:26 4:4 SyncActivityProfilerHandler.cpp:46] profiler_stop


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                               backward        27.09%      44.301ms        27.22%      44.510ms      11.127ms       0.000us         0.00%       7.679us       1.920us             4  
                                  cudaDeviceSynchronize        23.34%      38.167ms        23.34%      38.167ms      38.167ms       0.000us         0.00%       0.000us       0.000us             1  
         

In [9]:
# Is the loader the bottleneck? Measure it alone. Warm up first (worker start-up
# and the first reads are slow), and take the best of a few trials, because loader
# throughput on a shared or cloud CPU is noisy.
print("CPU cores available:", os.cpu_count())

def loader_rate(loader, n=30, warmup=10, trials=3):
    best = 0.0
    for _ in range(trials):
        it = iter(loader)
        for _ in range(warmup): next(it)      # spin up workers, fill the prefetch queue
        t0 = time.perf_counter()
        for _ in range(n): next(it)
        best = max(best, n / (time.perf_counter() - t0))
    return best

for w in [0, 8, 12, 16]:
    dl = DataLoader(ds, batch_size=64, num_workers=w, pin_memory=True,
                    persistent_workers=(w > 0),
                    prefetch_factor=(4 if w > 0 else None))
    print(f"num_workers={w:2d}: {loader_rate(dl):.1f} batches/s")

CPU cores available: 32


num_workers= 0: 11.0 batches/s


num_workers= 8: 22.2 batches/s


num_workers=12: 21.6 batches/s


num_workers=16: 31.0 batches/s


**Reading the sweep:** throughput is lowest at `num_workers=0`, because loading and compute cannot overlap. Adding workers overlaps data loading with GPU work and raises the rate until there are enough workers to keep the GPU fed, after which it flattens near the machine's core count. These numbers are noisy on a shared or cloud machine, so read the trend, not any single point. Explore next: does raising `batch_size` or dropping `pin_memory` move where the rate flattens?

## 6. Optimizer implementations
The same AdamW update, three implementations, on a model with many small parameter tensors.

**Look for:** the number of GPU bars per step drops sharply as you go down the list. `forloop` launches one small kernel per parameter per operation, so the GPU row is a long comb of tiny bars. `foreach` batches the same math across all tensors into a handful of `multi_tensor_apply` kernels. `fused` collapses the whole step into essentially one `_fused_adamw_` kernel. In the table this is the `# of Calls` column shrinking and `CUDA time avg` per kernel growing. Fewer, larger kernels means less launch overhead, which is why the step gets faster even though the arithmetic is identical.

In [10]:
# A model with many small weight matrices, so the optimizer has many tensors to update.
big = torch.nn.Sequential(*[torch.nn.Linear(1024, 1024) for _ in range(24)]).to(dev)
# Give every parameter a fake gradient so that optimizer.step() has real work to do.
for p in big.parameters(): p.grad = torch.randn_like(p)

# Three implementations of the SAME AdamW update:
#   forloop: loop over parameters in Python, one set of kernels per tensor (most launches)
#   foreach: one batched kernel across all tensors of a kind (fewer launches)
#   fused:   the whole update in a single kernel (fewest launches)
for label, kw in [("forloop", dict(foreach=False)), ("foreach", dict(foreach=True)), ("fused", dict(fused=True))]:
    o = torch.optim.AdamW(big.parameters(), lr=1e-4, **kw)
    print(f"{label:8s} {bench(o.step, 20):.3f} ms per step")
    run_profile(o.step, f"adamw_{label}", steps=4, rows=6)

forloop  6.977 ms per step


USDT:2026-09-28 04:10:15 4:4 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-09-28 04:10:15 4:4 SyncActivityProfilerHandler.cpp:46] profiler_stop


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                              Optimizer.step#AdamW.step         0.00%       0.000us         0.00%       0.000us       0.000us      54.490ms      1400.12%      54.490ms      27.245ms             2  
                                             aten::mul_         3.10%       1.825ms         6.27%       3.690ms      19.221us       1.071ms        27.53%       1.071ms       5.580us           192  
void at::

foreach  2.957 ms per step
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                              Optimizer.step#AdamW.step         0.00%       0.000us         0.00%       0.000us       0.000us       3.206ms       101.46%       3.206ms       1.603ms             2  
                                aten::_foreach_addcdiv_         0.67%      37.692us         1.04%      58.582us      29.291us     631.128us        19.97%     631.128us     315.564u

USDT:2026-09-28 04:10:16 4:4 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-09-28 04:10:16 4:4 SyncActivityProfilerHandler.cpp:46] profiler_stop
USDT:2026-09-28 04:10:17 4:4 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-09-28 04:10:17 4:4 SyncActivityProfilerHandler.cpp:46] profiler_stop


## 7. Memory snapshot
Record allocations for three training steps, then drag `mem.pickle` onto **pytorch.org/memory_viz**. Look for the parameter band, the optimizer state band after the first step, and one sawtooth per step.

In [11]:
# Start recording every GPU allocation and free. max_entries caps how much history we keep.
torch.cuda.memory._record_memory_history(max_entries=100_000)
step = make_step(loader_b)                     # reuse the fast input pipeline from section 5
for _ in range(3):                             # three steps is enough to see the pattern
    step()
torch.cuda.synchronize()                       # let all GPU work finish before we snapshot
# Write the recorded history to a file you drag onto pytorch.org/memory_viz.
torch.cuda.memory._dump_snapshot("mem.pickle")
torch.cuda.memory._record_memory_history(enabled=None)   # stop recording
# max_memory_allocated returns bytes; / 2**20 converts to MiB.
print("peak allocated:", torch.cuda.max_memory_allocated() / 2**20, "MiB")

peak allocated: 1945.5908203125 MiB


## 8. HTA: summarize a trace as numbers

Perfetto shows the timeline of one trace. **Holistic Trace Analysis (HTA)** reads the same trace files and answers questions with numbers:

- **Temporal breakdown:** what share of GPU time is compute, communication, memory copies, and idle.
- **Idle time breakdown:** when the GPU is idle, is it waiting for the CPU (host wait), waiting for another kernel, or other.
- **Kernel breakdown:** which kernels take the most GPU time.

HTA was built for jobs with many GPUs, so it expects a folder with one trace file per rank. We have one GPU, so each folder holds one file, rank 0.

We compare two pairs of traces from earlier sections:
1. Section 3: the Python loop against the vectorized version (launch bound).
2. Section 5: `num_workers=0` against 8 workers with pinned memory (input pipeline).

In [ ]:
# HTA is a separate package. -q keeps the install output short.
!pip install -q HolisticTraceAnalysis

In [ ]:
import json, shutil
from hta.trace_analysis import TraceAnalysis

def hta_for(name):
    """Copy traces/<name>.json into its own folder as rank 0 and load it with HTA."""
    folder = f"hta/{name}"
    shutil.rmtree(folder, ignore_errors=True)    # start clean if we re-run the cell
    os.makedirs(folder, exist_ok=True)
    with open(f"traces/{name}.json") as f:
        trace = json.load(f)
    # HTA reads the rank from distributedInfo. A single-GPU trace has none, so add it.
    trace.setdefault("distributedInfo", {"rank": 0, "world_size": 1})
    with open(f"{folder}/rank0.json", "w") as f:
        json.dump(trace, f)
    return TraceAnalysis(trace_dir=folder)

def summarize(name):
    an = hta_for(name)
    # visualize=False returns DataFrames instead of drawing plots, so we can print them.
    temporal = an.get_temporal_breakdown(visualize=False)
    idle = an.get_idle_time_breakdown(visualize=False)
    idle = idle[0] if isinstance(idle, tuple) else idle   # some versions return (df, stats)
    print(f"===== {name} =====")
    print("Temporal breakdown (share of GPU time):")
    print(temporal.to_string(index=False))
    print("\nIdle time breakdown, per stream:")
    print(idle.to_string(index=False))
    print()
    return an

### 8a. Launch bound: loop against vectorized

**Look for:** the loop trace has a large `idle_time_pctg` in the temporal breakdown, and most of that idle time is **host wait**: the GPU finishes each tiny kernel and waits for the CPU to launch the next one. The vectorized trace has a much smaller idle share.

In [ ]:
an_loop = summarize("launch_bound_loop")
an_vec  = summarize("launch_bound_vectorized")

### 8b. Input pipeline: 0 workers against 8 workers

**Look for:** with `num_workers=0` the GPU is idle for most of each step while the CPU loads the next batch, and that idle time is host wait. With 8 workers and pinned memory the idle share drops, because loading overlaps with GPU work.

In [ ]:
an_w0 = summarize("input_workers0")
an_w8 = summarize("input_workers8_pinned")

### 8c. Which kernels take the time

The kernel breakdown sorts kernels by total GPU time. Here it is for the 8-worker trace. With `visualize=True` HTA also draws pie charts like the ones on the HTA slide in the lecture.

In [ ]:
kernel_type_df, kernel_df = an_w8.get_gpu_kernel_breakdown(visualize=False)
print("By kernel type:")
print(kernel_type_df.to_string(index=False))
print("\nTop kernels:")
print(kernel_df.sort_values("sum", ascending=False).head(8).to_string(index=False))

In [ ]:
# Optional: draw the plots. HTA uses plotly, which renders inline in Colab and Jupyter.
an_w0.get_temporal_breakdown(visualize=True)
an_w0.get_idle_time_breakdown(visualize=True)

**Explore next:** run `summarize("resnet18_eager")` and `summarize("resnet18_compiled")` from section 2, and `summarize("sync_every_iter")` and `summarize("one_sync")` from section 4. Compare the idle share and how much of it is host wait. For a multi-GPU job, put one trace per rank in the same folder and HTA reports every rank side by side.

## What to report
Profiler times include profiler overhead. Use the `bench` numbers, measured without the profiler, for any speedup you report, and state the GPU, PyTorch version, batch size and number of runs.